# Release acceptance: provider → GEOID → overlay → durable output → statistics

## What this shows
Actual public Siege Utilities calls compose on deterministic synthetic data, with analytic expected outputs and negative paths. This is an offline acceptance lane, not certification of external services or production data.

## Why it matters
Provider adaptation, leading-zero join keys, area allocation, missingness, output preservation and statistical inference must agree across the release seams. A baseline run does not certify a later integrated candidate.

## Prereqs
Python >=3.11, siege-utilities, pandas<3, numpy, scipy, geopandas, shapely>=2, pyproj, pyarrow, nbclient and ipykernel. The runner records exact installed versions and candidate tree. Use the declared isolated environment; no credentials are required.

## Next
Rerun every cell from a fresh kernel after genuine candidate integration. Keep the source notebook output-clean; retain the executed notebook and manifest separately.


## Input and oracle contract
All shapes below are invented rectangles in EPSG:6933 (equal-area metres), not real Census boundaries. Source A occupies [0,2]×[0,2] with population 100 and rate 0.2; B occupies [2,4]×[0,2] with population 300 and rate 0.6. Targets L=[0,1], M=[1,3], R=[3,4] across the same height. Therefore populations must be [50,200,150], rates [0.2,0.4,0.6], total mass 400. These numbers are hand-derived from widths, not computed by the implementation under test. Explicit area_crs=6933 avoids an unrelated projection approximation. External Census HTTP transport is replaced by Census-format CSV passed through the real dependency parser. GADM dependency read_file URL acquisition is redirected to an actual local synthetic GeoJSON file; the actual GADM provider adapter, ID helpers, interpolation, statistics and atomic output code run unchanged. This does NOT exercise HTTP discovery, retries, remote downloads, credentials or service availability.

In [ ]:
import json
import csv
import io
import math
import sys
import importlib.metadata as metadata
import siege_utilities as su
import os
from pathlib import Path
from unittest.mock import patch
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from siege_utilities.geo.providers.boundary_providers import (
    resolve_boundary_provider, CensusTIGERProvider, GADMProvider, RDHProvider, BoundaryFetchError,
)
from siege_utilities.geo.boundary_result import BoundaryFetchResult, BoundaryInputError
from siege_utilities.geo.geoid_utils import (
    normalize_geoid, prepare_geoid_for_join, validate_geoid, geoid_to_slug, slug_to_geoid,
)
from siege_utilities.geo.interpolation import interpolate_areal
from siege_utilities.geo.spatial_transformations import convert_spatial_format
from siege_utilities.geo.providers.census_batch import CensusBatchGeocoder
from siege_utilities.geo.providers.census_geocoder import CensusVintage
from siege_utilities.geo.classification import classify_series, classify_choropleth
import censusgeocode
from pyproj import Transformer
from siege_utilities.data.statistics.moe_propagation import Estimate, moe_sum, moe_ratio, moe_proportion
from siege_utilities.data.statistics.cross_tabulation import contingency_table, rate_table, chi_square_test

# The runner verifies actual source/wheel bytes; component diagnostics have NO Git candidate SHA.
# Fail closed when manually run without declared provenance; never assume a candidate.
manifest_path = Path(os.environ['SU_ACCEPTANCE_MANIFEST'])
manifest = json.loads(manifest_path.read_text())
assert len(manifest['candidate_tree']) == 40
if manifest.get('diagnostic_only'):
    # A composed fixture has no Git commit; never relabel its base as candidate.
    assert manifest['candidate_sha'] is None
    assert manifest['diagnostic_id'] == os.environ['SU_ACCEPTANCE_DIAGNOSTIC_ID']
    assert manifest['base_sha'] == os.environ['SU_ACCEPTANCE_BASE_SHA']
    assert 'local component composition' in manifest['candidate_source_kind']
else:
    assert manifest['candidate_sha'] == os.environ['SU_ACCEPTANCE_CANDIDATE_SHA']
    assert len(manifest['candidate_sha']) == 40
assert su.__version__ == manifest['library_version']
assert Path(su.__file__).resolve().is_relative_to(Path(manifest['env_prefix']).resolve())
assert Path(sys.prefix).resolve() == Path(manifest['env_prefix']).resolve()
for package,version in manifest['key_dependencies'].items():
    assert metadata.version(package) == version,(package,version,metadata.version(package))
print(json.dumps({'candidate_sha':manifest['candidate_sha'],'candidate_tree':manifest['candidate_tree'],'diagnostic_only':manifest.get('diagnostic_only',False),'diagnostic_id':manifest.get('diagnostic_id'),'base_sha':manifest.get('base_sha'),'library_version':su.__version__,'library_origin':str(Path(su.__file__).resolve()),'python':sys.version,'key_dependencies':manifest['key_dependencies']},sort_keys=True))
OUTPUT = Path(os.environ['SU_ACCEPTANCE_OUTPUT']).resolve()
assert not OUTPUT.exists(), 'A fresh nonexistent OUTPUT directory is required per run'
OUTPUT.mkdir(parents=True)
print('Public acceptance imports resolved; fixtures are synthetic, external I/O substituted explicitly.')

def expect_error(label, exc_type, action, text):
    try:
        action()
    except exc_type as error:
        assert text.lower() in str(error).lower(), (label, str(error))
        print(json.dumps({'case': label, 'expected_error': exc_type.__name__, 'actual_error': type(error).__name__, 'message': str(error)}))
        return error
    raise AssertionError(f'{label}: expected {exc_type.__name__}; actual success')

In [ ]:
# Own transport fixture: real submitted CSV IDs are echoed in service-sorted order.
# No Siege function or dependency CSV parser is patched.
fixture_to_wgs84 = Transformer.from_crs(6933,4326,always_xy=True)
fixture_locations = {
    '10 Acceptance Way': (1.,1.,'000100'),
    '20 Acceptance Way': (3.,1.,'000200'),
    '30 Acceptance Way': (1.5,1.,'000100'),
    '40 Acceptance Way': (3.5,1.,'000200'),
}

def census_transport(*, drop_street=None, fail_street=None):
    def post(self,data=None,f=None,**kwargs):
        if f is not None:
            body = f.read()
            if isinstance(body,bytes):
                body = body.decode('utf-8')
            submitted = list(csv.reader(io.StringIO(body)))
        else:
            submitted = [[str(r['id']),r['street'],r.get('city',''),r.get('state',''),r.get('zip','')] for r in data]
        if any(r[1] == fail_street for r in submitted):
            raise OSError('synthetic transport failure for one chunk')
        response = io.StringIO()
        writer = csv.writer(response)
        for row in sorted(submitted,key=lambda r:r[0]):
            row_id,street = row[:2]
            if street == drop_street:
                continue
            if street == 'No Such Fixture':
                writer.writerow([row_id,street,'No_Match','','','','','','','','',''])
                continue
            x,y,tract = fixture_locations[street]
            lon,lat = fixture_to_wgs84.transform(x,y)
            writer.writerow([row_id,street,'Match','Exact',street,f'{lon},{lat}','100','L','06','037',tract,'1000'])
        return self._parse_batch_result(response.getvalue(),kwargs.get('returntype','geographies'))
    return post

address_input = pd.DataFrame([
    {'id':'item-b','street':'10 Acceptance Way','city':'Synthetic','state':'CA','zipcode':'90000'},
    {'id':'item-a','street':'20 Acceptance Way','city':'Synthetic','state':'CA','zipcode':'90000'},
])
with patch.object(censusgeocode.CensusGeocode,'_post_batch',new=census_transport()):
    geocoded = CensusBatchGeocoder(vintage=CensusVintage.CENSUS_2020).geocode(address_input.to_dict('records'))
geocoded_frame = geocoded.to_dataframe()
assert len(geocoded_frame) == len(address_input) == 2
assert geocoded_frame.input_id.tolist() == ['item-b','item-a']
assert geocoded_frame.tract_geoid.tolist() == ['06037000100','06037000200']
# Carry input position, never join on possibly duplicate/blank caller IDs.
geocoded_frame['input_position'] = np.arange(len(address_input))
assert geocoded_frame.input_position.is_unique
print(geocoded_frame[['input_position','input_id','tract_geoid','lat','lon']].to_json(orient='records'))


In [ ]:
fixture_boundaries = gpd.GeoDataFrame(
    {'GEOID': ['06037000100', '06037000200'], 'NAME': ['Synthetic A', 'Synthetic B']},
    geometry=[box(0,0,2,2), box(2,0,4,2)], crs='EPSG:6933', index=[17,4],
)
provider = resolve_boundary_provider('us')
assert isinstance(provider, CensusTIGERProvider)
assert isinstance(resolve_boundary_provider('PR'), CensusTIGERProvider)
assert isinstance(resolve_boundary_provider('DE'), GADMProvider)
assert 'tract' in provider.list_levels()
# This deliberately synthetic GADM response carries US-style IDs only to exercise
# the shared geometry contract; it is NOT a real German or Census dataset.
fixture_path = OUTPUT / 'synthetic-boundaries.geojson'
fixture_boundaries.to_file(fixture_path,driver='GeoJSON')
read_file = gpd.read_file
expected_url = 'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm4.1_DEU_1.json'
def acquire_local_boundary(url,*args,**kwargs):
    assert url == expected_url
    return read_file(fixture_path,*args,**kwargs)
with patch.object(gpd,'read_file',side_effect=acquire_local_boundary) as acquisition:
    boundaries = resolve_boundary_provider('DE').get_boundary('admin1',identifier='DEU')
    acquisition.assert_called_once_with(expected_url)
points = gpd.GeoSeries(gpd.points_from_xy(geocoded_frame.lon,geocoded_frame.lat),crs='EPSG:4326').to_crs(boundaries.crs)
assert all(boundaries.geometry.iloc[i].contains(points.iloc[i]) for i in range(2))
assert boundaries.GEOID.tolist() == ['06037000100', '06037000200']
assert boundaries.crs.to_epsg() == 6933
# Integral numeric IDs intentionally lose the first zero; library restores it before joining.
attributes = pd.DataFrame({'raw_id': [6037000100.0, 6037000200.0], 'population':[100.,300.], 'rate':[0.2,0.6]})
normalized = prepare_geoid_for_join(attributes, 'raw_id', 'tract', output_column='GEOID')
assert normalized.GEOID.tolist() == geocoded_frame.tract_geoid.tolist()
assert normalized.GEOID.tolist() == ['06037000100', '06037000200']
assert 'GEOID' not in attributes.columns
source = boundaries.merge(normalized[['GEOID','population','rate']], on='GEOID', validate='one_to_one')
source.index = [17,4]
assert source.population.tolist() == [100.,300.]
assert not validate_geoid('6037000100', 'tract')
assert validate_geoid('06037000100', 'tract')
assert slug_to_geoid(geoid_to_slug('06037000100','tract')) == '06037000100'
assert slug_to_geoid(geoid_to_slug('50F-3','sldl')) == '50F-3'
expect_error('fractional ID', ValueError, lambda: normalize_geoid(6037.5,'county'), 'integral')
expect_error('invalid state prefix', ValueError, lambda: normalize_geoid('99037','county'), 'state')
print(source.drop(columns='geometry').to_json(orient='records'))
# Mixed source representations preserve null and normalize both numeric/string keys.
mixed = prepare_geoid_for_join(pd.DataFrame({'key':[6037,'06037',None]}),'key','county')
assert mixed.key.iloc[:2].tolist() == ['06037','06037'] and pd.isna(mixed.key.iloc[2])
expect_error('implausibly short tract ID',ValueError,lambda: normalize_geoid('6037','tract'),'state')
duplicate_attributes = pd.concat([normalized,normalized.iloc[[0]]],ignore_index=True)
expect_error('duplicate source join key',pd.errors.MergeError,lambda: boundaries.merge(duplicate_attributes,on='GEOID',validate='one_to_one'),'unique')


In [ ]:
targets = gpd.GeoDataFrame({'zone':['L','M','R'], 'note':['left','middle','right'], 'synthetic_zone_id':['001','002','003']}, geometry=[box(0,0,1,2),box(1,0,3,2),box(3,0,4,2)], crs='EPSG:6933', index=[91,12,55])
result = interpolate_areal(source, targets, extensive_variables=['population'], intensive_variables=['rate'], allocate_total=False, area_crs='EPSG:6933')
np.testing.assert_allclose(result.data.population, [50,200,150], rtol=1e-6, atol=1e-6)
np.testing.assert_allclose(result.data.rate, [0.2,0.4,0.6], rtol=1e-6, atol=1e-6)
assert math.isclose(result.data.population.sum(),400,rel_tol=1e-6)
assert result.data.index.tolist() == [91,12,55]
assert result.data.zone.tolist() == ['L','M','R']
assert result.data.note.tolist() == ['left','middle','right']
assert result.data.geometry.equals(targets.geometry)
assert result.data.crs == source.crs
assert result.n_source == 2 and result.n_target == 3
print(json.dumps({'case':'full overlay','backend':result.backend,'area_crs':result.area_crs,'expected_population':[50,200,150],'actual_population':result.data.population.tolist(),'expected_rate':[0.2,0.4,0.6],'actual_rate':result.data.rate.tolist()}))
assert result.backend == manifest['declared_backend'],(result.backend,manifest['declared_backend'])


## Missingness and backend variants
An intensive value of zero is observed data, not missing data. A target with no positive-area contributor (including an edge-only touch) has population zero but undefined rate (NaN). A missing extensive contributor makes the target total unknown; a missing intensive contributor is excluded from its own valid-area denominator. Partial coverage with allocate_total=False allocates only covered mass. With True, the documented Tobler branch redistributes all source mass to overlapping targets; Shapely/DuckDB retain the covered fraction. Record the selected backend and do not claim interchangeable semantics.
Empty-source extensive zero is the documented current interpolation policy, not proof that an empty data-acquisition result means observed zero; this policy has an SU-1 ambiguity and must never hide an acquisition error.

Release evidence requires TWO fresh environment lanes: Shapely-only (no Tobler or DuckDB) and Tobler with mapclassify. allocate_total=True remains a documented backend limitation, NOT an equivalence claim. Strict classification fallback equivalence is separately asserted in the red section.


In [ ]:
edge_targets = gpd.GeoDataFrame({'zone':['A','B','gap','touch']}, geometry=[box(0,0,2,2),box(2,0,4,2),box(10,0,11,2),box(4,0,5,2)], crs='EPSG:6933', index=[9,2,8,7])
missing_source = source.copy()
missing_source['population'] = [100.,np.nan]
missing_source['rate'] = [0.,np.nan]
missing = interpolate_areal(missing_source, edge_targets, ['population'], ['rate'], allocate_total=False, area_crs='EPSG:6933')
np.testing.assert_allclose(missing.data.population,[100,np.nan,0,0],equal_nan=True,atol=1e-6)
np.testing.assert_allclose(missing.data.rate,[0,np.nan,np.nan,np.nan],equal_nan=True,atol=1e-6)
wide = gpd.GeoDataFrame({'zone':['both']}, geometry=[box(0,0,4,2)], crs='EPSG:6933')
valid_rate = interpolate_areal(missing_source,wide,intensive_variables=['rate'],area_crs='EPSG:6933')
assert valid_rate.data.rate.iloc[0] == 0.
partial = targets.iloc[[0]].copy()
partial_false = interpolate_areal(source,partial,['population'],['rate'],allocate_total=False,area_crs='EPSG:6933')
partial_true = interpolate_areal(source,partial,['population'],['rate'],allocate_total=True,area_crs='EPSG:6933')
np.testing.assert_allclose(partial_false.data.population,[50],atol=1e-6)
expected_true = 100. if partial_true.backend == 'tobler' else 50.
np.testing.assert_allclose(partial_true.data.population,[expected_true],atol=1e-6)
empty_result = interpolate_areal(source.iloc[:0],edge_targets,['population'],['rate'],area_crs='EPSG:6933')
np.testing.assert_allclose(empty_result.data.population,[0,0,0,0])
assert empty_result.data.rate.isna().all()
print(json.dumps({'case':'missingness and partial coverage','backend':missing.backend,'population':missing.data.population.tolist(),'rate':missing.data.rate.tolist(),'partial_false':partial_false.data.population.tolist(),'partial_true':partial_true.data.population.tolist(),'expected_partial_true':expected_true}))

In [ ]:
# Different input CRS, explicit metric output; compare with analytic baseline.
reprojected = interpolate_areal(source,targets.to_crs('EPSG:4326'),['population'],['rate'],allocate_total=False,crs='EPSG:6933',area_crs='EPSG:6933')
np.testing.assert_allclose(reprojected.data.population,[50,200,150],rtol=1e-5,atol=1e-5)
assert reprojected.data.crs.to_epsg() == 6933
assert reprojected.warnings
expect_error('invalid area CRS',ValueError,lambda: interpolate_areal(source,targets,['population'],area_crs='EPSG:4326'),'geographic')
expect_error('missing variable',ValueError,lambda: interpolate_areal(source,targets,['not_present']),'not found')
expect_error('empty targets',ValueError,lambda: interpolate_areal(source,targets.iloc[:0],['population']),'empty')
expect_error('no variables',ValueError,lambda: interpolate_areal(source,targets),'variable')
expect_error('missing GADM country',ValueError,lambda: GADMProvider().get_boundary('admin1'),'country')
expect_error('invalid RDH level',ValueError,lambda: RDHProvider(username='',password='').get_boundary('not_a_level',identifier='TX'),'level')
failed = BoundaryFetchResult.fail('FIXTURE_FAILURE','input_validation','synthetic invalid input',{'year':2020})
expect_error('typed boundary failure',BoundaryInputError,failed.raise_on_error,'synthetic')
# Acquisition failure propagates through the actual GADM public method.
with patch.object(gpd,'read_file',side_effect=OSError('synthetic acquisition unavailable')):
    expect_error('provider acquisition failure',OSError,lambda: GADMProvider().get_boundary('admin1',identifier='DEU'),'unavailable')
print('Explicit CRS and failure-path assertions passed.')

In [ ]:
# Shared release contract is upper-inclusive in EVERY backend.
# The old NumPy behavior is a baseline failure, never an adaptive expectation.
classified = classify_choropleth(result.data,'population',scheme='equal_interval',k=2)
assert classified.zone.tolist() == ['L','M','R']
assert classified.geometry.equals(result.data.geometry)
assert classified['_bin'].tolist() == [0,1,1]
assert classified['_color'].str.fullmatch(r'#[0-9a-fA-F]{6}').all()
# Render actual geographic output rather than only a mock report payload.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
figure,axis = plt.subplots(figsize=(5,2))
classified.plot(ax=axis,color=classified['_color'],edgecolor='black')
axis.set_title('Synthetic population allocation: 50 / 200 / 150')
figure.savefig(OUTPUT/'overlay-map.png',dpi=120)
plt.close(figure)
assert (OUTPUT/'overlay-map.png').stat().st_size > 1000
edge = classify_series([0.,1.,2.,3.,4.,np.nan],scheme='equal_interval',k=2)
expected = [0,0,0,1,1,-1]
assert edge.bins.tolist() == expected
print(json.dumps({'case':'classification exact interior edge','backend':edge.backend,'breaks':edge.breaks.tolist(),'expected_bins':expected,'actual_bins':edge.bins.tolist(),'shared_edge_contract_checked':True}))


In [ ]:
# Real library atomic GeoJSON output and real GeoPandas reads; no output mocks.
geojson_path = OUTPUT / 'overlay.geojson'
convert_spatial_format(result.data,'geojson',output_path=str(geojson_path))
loaded = gpd.read_file(geojson_path)
assert loaded.zone.tolist() == ['L','M','R']
assert loaded.synthetic_zone_id.tolist() == ['001','002','003']
assert all(isinstance(value,str) for value in loaded.synthetic_zone_id)
np.testing.assert_allclose(loaded.population,[50,200,150])
np.testing.assert_allclose(loaded.rate,[0.2,0.4,0.6])
assert loaded.crs == result.data.crs
assert loaded.geometry.geom_equals_exact(result.data.geometry.reset_index(drop=True), tolerance=1e-8).all()
# GeoParquet is native GeoPandas I/O, explicitly not claimed as a Siege helper.
assert loaded.index.tolist() == [0,1,2], 'GeoJSON does not preserve source row labels'
parquet_path = OUTPUT / 'overlay.parquet'
result.data.to_parquet(parquet_path)
parquet = gpd.read_parquet(parquet_path)
pd.testing.assert_frame_equal(parquet.drop(columns='geometry'),result.data.drop(columns='geometry'))
assert parquet.crs == result.data.crs
assert parquet.geometry.equals(result.data.geometry)
expect_error('unsupported output',ValueError,lambda: convert_spatial_format(result.data,'not_a_format',output_path=str(OUTPUT/'never')),'unsupported')
print(json.dumps({'case':'durable output','geojson_features':len(loaded),'parquet_features':len(parquet),'population_total':float(parquet.population.sum())}))
assert not (OUTPUT/'never').exists()

# Inject a third-party filesystem writer failure after it creates temporary data.
# The public SU conversion and atomic_write_path implementation remain unchanged.
failed_destination=OUTPUT/'failed-overlay.geojson'
before_files={p.name for p in OUTPUT.iterdir()}
def fail_dependency_write(self,filename,*args,**kwargs):
    Path(filename).write_text('synthetic partial bytes',encoding='utf-8')
    raise OSError('synthetic disk-full output failure')
with patch.object(gpd.GeoDataFrame,'to_file',new=fail_dependency_write):
    expect_error('atomic output rollback',OSError,lambda: convert_spatial_format(result.data,'geojson',output_path=str(failed_destination)),'disk-full')
assert not failed_destination.exists()
assert {p.name for p in OUTPUT.iterdir()} == before_files,'atomic failure left temporary data'


## Statistical composition and expected results
Source estimates 100±6 and 300±8 sum to 400±10 under independent-estimate MOE quadrature. These are illustrative uncertainty inputs, not an inference that areal interpolation preserves independent errors. The interpolated population total supplies the denominator of a separate synthetic proportion calculation. An independent balanced 2×2 respondent fixture has 10 observations per cell: all row rates 0.5, chi-square statistic 0, p=1, dof=1. Zero denominators and invalid normalization must raise, not yield a success-shaped result.

In [ ]:
combined = moe_sum([Estimate(value=100,moe=6),Estimate(value=300,moe=8)])
assert math.isclose(combined.value,float(parquet.population.sum()),rel_tol=1e-6)
assert combined.value == 400.
assert math.isclose(combined.moe,10.)
assert combined.confidence_interval() == (390.,410.)
proportion = moe_proportion(Estimate(value=100,moe=6),combined)
assert proportion.value == 0.25
assert math.isclose(proportion.moe, math.sqrt(36 - 0.25**2 * 100)/400,rel_tol=1e-12)
respondents = pd.DataFrame([(a,b) for a in ['A','B'] for b in ['yes','no'] for _ in range(10)],columns=['group','answer'])
ct = contingency_table(respondents,'group','answer',margins=True)
np.testing.assert_allclose(ct.loc[['A','B'],['no','yes']],[[10,10],[10,10]])
rates = rate_table(ct,normalize='index')
np.testing.assert_allclose(rates,[[0.5,0.5],[0.5,0.5]])
chi = chi_square_test(ct)
assert chi.statistic == 0. and chi.p_value == 1. and chi.dof == 1
np.testing.assert_allclose(chi.expected_freq,[[10,10],[10,10]])
expect_error('MOE zero denominator',ZeroDivisionError,lambda: moe_ratio(Estimate(1,1),Estimate(0,1)),'zero')
expect_error('invalid normalization',ValueError,lambda: rate_table(ct,'invalid'),'normalize')
expect_error('zero contingency',ZeroDivisionError,lambda: rate_table(pd.DataFrame([[0,0],[0,0]])),'zero')
expect_error('unsupported confidence',ValueError,lambda: combined.confidence_interval(0.8),'confidence')
print(json.dumps({'case':'statistics','sum':combined.value,'moe':combined.moe,'proportion':proportion.value,'proportion_moe':proportion.moe,'chi_square':chi.statistic,'p_value':chi.p_value,'dof':chi.dof}))
fallback = moe_proportion(Estimate(50,1),Estimate(100,20))
assert math.isclose(fallback.moe,math.sqrt(1 + 0.5**2 * 20**2)/100,rel_tol=1e-12)

# Nontrivial chi-square, Yates-corrected as used by scipy's default for dof=1.
# Expected count 15/cell; each corrected deviation is 4.5: 4*(4.5**2)/15=5.4.
nontrivial = pd.DataFrame([[20,10],[10,20]],index=['A','B'],columns=['yes','no'])
nontrivial_chi = chi_square_test(nontrivial)
assert math.isclose(nontrivial_chi.statistic,5.4,abs_tol=1e-6)
assert 0.02 < nontrivial_chi.p_value < 0.021
assert nontrivial_chi.dof == 1
expect_error('zero-row chi-square',ValueError,lambda: chi_square_test(pd.DataFrame([[0,0],[10,20]])),'zero')
print(json.dumps({'case':'nontrivial Yates chi-square','expected_statistic':5.4,'actual_statistic':nontrivial_chi.statistic,'p_value':nontrivial_chi.p_value}))


## Release seam assertions (remain red until the candidate really repairs them)
The following independent public-call checks enforce the requested release behavior, not the current baseline's defects. They are not skipped or xfailed. A complete diagnostic baseline may execute with allow_errors=True to collect every failing cell, but its overall verdict MUST fail if any code cell fails. A release acceptance run must have no failing cells. The declared runner retains all code outputs and expected-versus-actual observations. #1824 and #1826 heads alone are not integrated develop candidates.

Analytics expected behavior is derived from issue #1422 criteria (https://github.com/siege-analytics/siege_utilities/issues/1422), not evidence that a particular unmerged PR is accepted. Independent arithmetic: 5 won/1 lead=5, empty denominator undefined; MAPE of +10% and -10%=10%, mean bias0; weighted [5,1] at weights[3,1]=(15+1)/4=4. OLS uses all sorted equally spaced periods. Final integration requires rerunning these same frozen oracles against the actual repaired source.

Oracle provenance: #1422 supplies the mathematical corrections (snapshot ratios, finite-input grading, rounded tiers, normalized weights and respondent ordering). The strict >30-day default aging boundary and inclusive ±1-percentage-point OLS stability band are explicit #1826 design/implementation contract pins, not requirements stated by #1422 itself: https://github.com/siege-analytics/siege_utilities/pull/1826. OLS uses all valid lexicographically sorted, equally spaced periods under that design.


In [ ]:
identity_input = [
    {'id':'same','street':'10 Acceptance Way'},
    {'id':'same','street':'20 Acceptance Way'},
    {'id':'','street':'30 Acceptance Way'},
    {'id':'','street':'40 Acceptance Way'},
]
with patch.object(censusgeocode.CensusGeocode,'_post_batch',new=census_transport()):
    identity_result = CensusBatchGeocoder(vintage=CensusVintage.CENSUS_2020).geocode(identity_input)
identity_frame = identity_result.to_dataframe()
expected_tracts = ['06037000100','06037000200','06037000100','06037000200']
actual_tracts = identity_frame.tract_geoid.tolist() if 'tract_geoid' in identity_frame else []
print(json.dumps({'case':'duplicate and blank caller IDs','expected_tracts':expected_tracts,'actual_tracts':actual_tracts}))
assert len(identity_frame) == 4
assert identity_frame.input_id.tolist() == ['same','same','','']
assert actual_tracts == expected_tracts
expected_xy = np.array([[1.,1.],[3.,1.],[1.5,1.],[3.5,1.]])
identity_points = gpd.GeoSeries(gpd.points_from_xy(identity_frame.lon,identity_frame.lat),crs=4326).to_crs(6933)
np.testing.assert_allclose(np.column_stack([identity_points.x,identity_points.y]),expected_xy,atol=1e-6)
identity_frame['input_position'] = range(4)
assert identity_frame.input_position.is_unique


In [ ]:
with patch.object(censusgeocode.CensusGeocode,'_post_batch',new=census_transport(drop_street='20 Acceptance Way')):
    dropped = CensusBatchGeocoder().geocode(address_input.to_dict('records'))
dropped_quality = [r.match_quality for r in dropped.results]
print(json.dumps({'case':'dropped response is error not no_match','expected_quality':['exact','error'],'actual_quality':dropped_quality,'errors':dropped.errors}))
assert len(dropped.results) == 2
assert dropped_quality == ['exact','error']
assert dropped.results[1].lat is None and dropped.results[1].lon is None


In [ ]:
chunk_input = [{'id':str(i),'street':street} for i,street in enumerate(['10 Acceptance Way','20 Acceptance Way','30 Acceptance Way'])]
with patch.object(censusgeocode.CensusGeocode,'_post_batch',new=census_transport(fail_street='20 Acceptance Way')):
    chunked = CensusBatchGeocoder(chunk_size=1).geocode(chunk_input)
actual_quality = [r.match_quality for r in chunked.results]
print(json.dumps({'case':'failed middle chunk preserves first and last','expected_quality':['exact','error','exact'],'actual_quality':actual_quality,'errors':chunked.errors}))
assert len(chunked.results) == 3
assert actual_quality == ['exact','error','exact']
assert chunked.results[0].tract_geoid == '06037000100'
assert chunked.results[2].tract_geoid == '06037000100'
assert chunked.errors


In [ ]:
no_match_input = [{'id':'same','street':'No Such Fixture'},{'id':'same','street':'10 Acceptance Way'}]
with patch.object(censusgeocode.CensusGeocode,'_post_batch',new=census_transport()):
    no_match = CensusBatchGeocoder().geocode(no_match_input)
position_frame = no_match.to_dataframe()
position_frame['input_position'] = np.arange(len(no_match_input))
print(json.dumps({'case':'no-match row retains input position','expected_quality':['no_match','exact'],'actual_quality':position_frame.match_quality.tolist()}))
assert position_frame.match_quality.tolist() == ['no_match','exact']
matched_positions = position_frame.loc[position_frame.matched,'input_position'].tolist()
assert matched_positions == [1]
assert not no_match.errors


In [ ]:
from datetime import datetime,timedelta,timezone
from siege_utilities.analytics.pipeline import PipelineAnalyzer,PipelineItem
as_of = datetime(2026,1,1,tzinfo=timezone.utc)
items = [PipelineItem('lead','lead',100,as_of-timedelta(days=30.1))] + [PipelineItem(f'won-{i}','won',20,as_of-timedelta(days=1)) for i in range(5)]
report = PipelineAnalyzer(['lead','won','empty','later']).analyze(items,as_of=as_of)
actual_ratios = [r.ratio for r in report.stage_ratios] if hasattr(report,'stage_ratios') else {'legacy_conversions':[r.rate for r in report.conversions]}
print(json.dumps({'case':'independent snapshot stage ratios','expected':[5.0,0.0,None],'actual':actual_ratios,'aging_expected':['lead'],'aging_actual':report.velocity[0].aging_items}))
assert hasattr(report,'stage_ratios'),'release requires stage_ratios, not legacy cohort-named conversions'
assert actual_ratios == [5.0,0.0,None]
assert report.velocity[0].aging_items == ['lead']

# #1826 design pin: default 30 days; alert strictly above, not at threshold.
aging_items=[PipelineItem(label,'lead',1,as_of-timedelta(days=age)) for label,age in [('under',29.9),('at',30.),('over',30.1)]]
aging_report=PipelineAnalyzer(['lead']).analyze(aging_items,as_of=as_of)
assert aging_report.velocity[0].aging_items == ['over']


In [ ]:
from siege_utilities.analytics.forecast import ForecastAnalyzer
forecast = ForecastAnalyzer([100.,200.],[110.,180.]).analyze().overall
assert forecast.mape == 10. and forecast.bias == 0.
insufficient = ForecastAnalyzer([100.,float('nan')],[110.,200.]).analyze().overall
print(json.dumps({'case':'nonfinite forecast','expected_grade':'Insufficient data','actual_grade':insufficient.grade,'expected_mape':None,'actual_mape':insufficient.mape}))
assert insufficient.grade == 'Insufficient data'
assert insufficient.mape is None and insufficient.bias is None
assert insufficient.bias_direction == 'insufficient-data'
# Same values shuffled: periods represent equal temporal spacing in this API.
trend = ForecastAnalyzer([100.]*4,[120.,100.,150.,120.],periods=['2026-04','2026-01','2026-02','2026-03']).analyze()
assert [p.period for p in trend.trend] == ['2026-01','2026-02','2026-03','2026-04']

assert trend.trend_direction == 'degrading', 'OLS on sorted MAPEs [0,50,20,20] has positive slope3, fitted total change9'


In [ ]:
from siege_utilities.analytics.comparison import ComparativeAnalyzer,DimensionScore
scores = {'A':{'quality':DimensionScore(5,'synthetic fixture A'),'cost':DimensionScore(1,'synthetic fixture A')},'B':{'quality':DimensionScore(1,'synthetic fixture B'),'cost':DimensionScore(5,'synthetic fixture B')}}
weighted = ComparativeAnalyzer(['quality','cost'],weights={'quality':3.,'cost':1.}).compare(scores)
unit_weighted = ComparativeAnalyzer(['quality','cost'],weights={'quality':0.75,'cost':0.25}).compare(scores)
print(json.dumps({'case':'weight scaling invariant','expected_totals':{'A':4.,'B':2.},'actual_totals':weighted.weighted_totals,'unit_totals':unit_weighted.weighted_totals}))
assert weighted.weighted_totals == unit_weighted.weighted_totals == {'A':4.,'B':2.}
expect_error('negative comparison weight',ValueError,lambda: ComparativeAnalyzer(['quality','cost'],weights={'quality':-1.,'cost':2.}),'weight')
expect_error('nonfinite comparison weight',ValueError,lambda: ComparativeAnalyzer(['quality','cost'],weights={'quality':float('inf'),'cost':1.}),'weight')

large = ComparativeAnalyzer(['quality','cost'],weights={'quality':1e308,'cost':1e308}).compare(scores)
assert large.weighted_totals == {'A':3.,'B':3.}, 'finite huge weights must normalize without overflow'


In [ ]:
from siege_utilities.geo.providers.batch_geocoder import BatchGeocoder,BatchGeocodingResult,GeocodingResult
from siege_utilities.geo.providers.composite_geocoder import CompositeBatchGeocoder
# A test-owned provider implementation, not a patched library response. It honors
# submitted IDs but returns its own service rows in reverse order.
class ReversingFixtureBackend(BatchGeocoder):
    @property
    def backend_name(self):
        return 'synthetic-reversing-provider'
    def geocode(self,addresses):
        rows=[]
        for address in addresses:
            x,y,tract=fixture_locations[address.street]
            lon,lat=fixture_to_wgs84.transform(x,y)
            rows.append(GeocodingResult(input_id=address.input_id,input_address=address.one_line,matched_address=address.street,lat=lat,lon=lon,tract_geoid='06037'+tract,match_quality='exact',backend=self.backend_name))
        return BatchGeocodingResult(results=list(reversed(rows)),backend=self.backend_name)
composite=CompositeBatchGeocoder([ReversingFixtureBackend()]).geocode(identity_input)
actual=[r.tract_geoid for r in composite.results]
expected=['06037000100','06037000200','06037000100','06037000200']
print(json.dumps({'case':'composite reversed provider results','expected':expected,'actual':actual}))
assert [r.input_id for r in composite.results] == ['same','same','','']
assert actual == expected


In [ ]:
class WrongIdFixtureBackend(BatchGeocoder):
    @property
    def backend_name(self):
        return 'synthetic-wrong-id-provider'
    def geocode(self,addresses):
        return BatchGeocodingResult(results=[GeocodingResult(input_id='UNRELATED-ID',lat=1.,lon=2.,match_quality='exact',backend=self.backend_name) for _ in addresses],backend=self.backend_name)
wrong_ids=CompositeBatchGeocoder([WrongIdFixtureBackend()]).geocode(address_input.to_dict('records'))
qualities=[r.match_quality for r in wrong_ids.results]
print(json.dumps({'case':'composite rejects wrong IDs','expected':['error','error'],'actual':qualities,'errors':wrong_ids.errors}))
assert len(wrong_ids.results)==2
assert qualities==['error','error']
assert all(r.lat is None and r.lon is None for r in wrong_ids.results)


In [ ]:
# Only dependency absence is simulated in a fresh interpreter. No SU state is patched.
import subprocess
numpy_probe=OUTPUT/'classification_numpy_probe.py'
numpy_probe.write_text("import sys,json\nsys.modules['mapclassify']=None\nfrom siege_utilities.geo.classification import classify_series\nr=classify_series([0.,1.,2.,3.,4.,float('nan')],scheme='equal_interval',k=2)\nprint(json.dumps({'backend':r.backend,'bins':r.bins.tolist(),'breaks':r.breaks.tolist()}))\n")
process=subprocess.run([sys.executable,'-I',str(numpy_probe)],check=True,capture_output=True,text=True,timeout=60)
numpy_edge=json.loads(process.stdout.splitlines()[-1])
upper_inclusive=[0,0,0,1,1,-1]
print(json.dumps({'case':'numpy backend must honor shared upper-inclusive edge contract','expected':upper_inclusive,'actual':numpy_edge}))
assert numpy_edge['backend']=='numpy'
assert numpy_edge['bins']==upper_inclusive


In [ ]:
from siege_utilities.analytics.scoring import Dimension,HealthScorer
from siege_utilities.analytics.risk import RiskSignal,SignalType,RiskTierConfig,RiskAnalyzer,RiskTier
health=HealthScorer([Dimension('observed',100)]).score('synthetic',{'observed':69.999})
risk=RiskAnalyzer([RiskSignal('observed',SignalType.THRESHOLD,100)],tier_config=RiskTierConfig(critical_min=90,high_min=70,medium_min=40),interventions={RiskTier.HIGH:'review'}).assess('synthetic',{'observed':69.999})
print(json.dumps({'case':'reported rounded score drives tier','expected_score':70.,'actual_health_score':health.composite_score,'actual_health_tier':health.tier,'actual_risk_score':risk.composite_score,'actual_risk_tier':risk.tier.value,'actual_intervention':risk.intervention}))
assert health.composite_score==70. and health.tier=='Green'
assert risk.composite_score==70. and risk.tier==RiskTier.HIGH and risk.intervention=='review'


In [ ]:
from siege_utilities.analytics.pricing import analyze_price_sensitivity
valid_prices=analyze_price_sensitivity([1.,2.],[3.,4.],[6.,7.],[9.,10.],min_sample=1)
assert valid_prices.sample_size==2
assert valid_prices.price_grid[0]==1. and valid_prices.price_grid[-1]==10.
expect_error('respondent price ordering',ValueError,lambda: analyze_price_sensitivity([5.],[4.],[6.],[8.],min_sample=1),'ordering at response index 0')


In [ ]:
summary = {'status':'final cell reached; runner must still reject ANY failed cell','backend':result.backend,'synthetic_inputs':True,'external_acquisition_substituted':True,'remote_services_certified':False,'library_calls':['CensusBatchGeocoder.geocode','CompositeBatchGeocoder.geocode','resolve_boundary_provider','GADMProvider.get_boundary','prepare_geoid_for_join','interpolate_areal','classify_series','classify_choropleth','convert_spatial_format','moe_sum','moe_proportion','contingency_table','rate_table','chi_square_test','PipelineAnalyzer.analyze','ForecastAnalyzer.analyze','ComparativeAnalyzer.compare','HealthScorer.score','RiskAnalyzer.assess','analyze_price_sensitivity']}
(OUTPUT/'assertions.json').write_text(json.dumps(summary,indent=2)+'\n',encoding='utf-8')
print(json.dumps(summary,sort_keys=True))


## Related
- Source: siege_utilities/geo/providers/boundary_providers.py; geo/geoid_utils.py; geo/interpolation/areal.py; geo/spatial_transformations.py; data/statistics/moe_propagation.py and cross_tabulation.py.
- Project tests: tests/test_notebook_hygiene.py, tests/test_notebooks.py; scripts/check_notebook_inventory.py.
- Architecture: docs/ARCHITECTURE.md; CLAUDE.md canonical composition and provider contracts.
- Execution evidence: candidate-pinned manifest, executed notebook, stdout/stderr logs and output hashes produced outside governed notebook source. A later candidate requires a new clean-kernel run.
